<a href="https://colab.research.google.com/github/mithun-martin/PYTORCH_TUTORIALS/blob/main/2_autograd.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# For tips on running notebooks in Google Colab, see
# https://docs.pytorch.org/tutorials/beginner/colab
%matplotlib inline

A Gentle Introduction to `torch.autograd`
=========================================

`torch.autograd` is PyTorch's automatic differentiation engine that
powers neural network training. In this section, you will get a
conceptual understanding of how autograd helps a neural network train.

Background
----------

Neural networks (NNs) are a collection of nested functions that are
executed on some input data. These functions are defined by *parameters*
(consisting of weights and biases), which in PyTorch are stored in
tensors.

Training a NN happens in two steps:

**Forward Propagation**: In forward prop, the NN makes its best guess
about the correct output. It runs the input data through each of its
functions to make this guess.

**Backward Propagation**: In backprop, the NN adjusts its parameters
proportionate to the error in its guess. It does this by traversing
backwards from the output, collecting the derivatives of the error with
respect to the parameters of the functions (*gradients*), and optimizing
the parameters using gradient descent. For a more detailed walkthrough
of backprop, check out this [video from
3Blue1Brown](https://www.youtube.com/watch?v=tIeHLnjs5U8).

Usage in PyTorch
----------------

Let\'s take a look at a single training step. For this example, we load
a pretrained resnet18 model from `torchvision`. We create a random data
tensor to represent a single image with 3 channels, and height & width
of 64, and its corresponding `label` initialized to some random values.
Label in pretrained models has shape (1,1000).

<div style="background-color: #54c7ec; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>NOTE:</strong></div>

<div style="background-color: #f3f4f7; padding-left: 10px; padding-top: 10px; padding-bottom: 10px; padding-right: 10px">

<p>This tutorial works only on the CPU and will not work on GPU devices (even if tensors are moved to CUDA).</p>

</div>



In [1]:
import torch
from torchvision.models import resnet18, ResNet18_Weights
model = resnet18(weights = ResNet18_Weights.DEFAULT)
data = torch.rand(1,3,64,64)
labels = torch.rand(1,1000)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 167MB/s]


**1. data = torch.rand(1, 3, 64, 64)**

torch.rand: This function creates a tensor filled with random numbers between 0 and 1. This is common for generating dummy data for testing purposes.

(1, 3, 64, 64): This tuple defines the shape of the tensor, and for image data, it typically represents:

**1:The batch size.** This means we're processing a single image at a time. In real-world scenarios, you'd usually have a larger batch size (e.g., 32, 64, 128) for efficiency.

**3: The number of color channels.** This is indeed for RGB (Red, Green, Blue) images, where each pixel has three values representing the intensity of each color component.

**64, 64**: **The height and width of the image in pixels**
So, it's a 64x64 pixel image.
So, torch.rand(1, 3, 64, 64) creates a placeholder for one 64x64 pixel RGB image, filled with random pixel values. It's a typical way to generate dummy input for an image model when you don't have actual image files handy for an example.

**2. labels = torch.rand(1, 1000)**

Why 1000?: The crucial part here is that the example uses a pre-trained resnet18 model from torchvision. These models are often pre-trained on large datasets like ImageNet, which has 1000 different object categories (classes). The output layer of such a model is designed to produce 1000 probabilities, one for each class.
(1, 1000): Similar to the data, 1 is the batch size (one set of labels for one image). 1000 corresponds to the number of possible classes the model was trained to predict.
torch.rand for labels: In a real scenario, labels would be one-hot encoded (e.g., [0, 0, 1, 0, ...] indicating the third class) or simply an integer representing the correct class ID. However, for this demonstration of how autograd works, torch.rand(1, 1000) is used to create a dummy target label tensor with random values, just to have something to compare the prediction against and calculate a loss. It's not a meaningful label in terms of classification but serves the purpose of showing the backward pass calculation.

Next, we run the input data through the model through each of its layers
to make a prediction. This is the **forward pass**.


In [2]:
prediction = model(data) # forward pass

We use the model\'s prediction and the corresponding label to calculate
the error (`loss`). The next step is to backpropagate this error through
the network. Backward propagation is kicked off when we call
`.backward()` on the error tensor. Autograd then calculates and stores
the gradients for each model parameter in the parameter\'s `.grad`
attribute.


In [3]:
loss = (prediction - labels).sum()
'''.sum(): This method calculates the sum of all elements in the resulting tensor from the subtraction.
By summing the differences, you get a single scalar value that represents the total 'error' or 'loss' of the
model's prediction compared to the true label. While often loss functions in deep learning use
more complex calculations (like Mean Squared Error, Cross-Entropy Loss, etc.),
for a simple demonstration of autograd,
a basic sum of differences serves to create a scalar loss value that can be backpropagated.'''
loss.backward() # backward pass

Next, we load an optimizer, in this case SGD with a learning rate of
0.01 and
[momentum](https://medium.com/data-science/stochastic-gradient-descent-with-momentum-a84097641a5d)
of 0.9. We register all the parameters of the model in the optimizer.


In [4]:
# optim = torch.optim.SGD(model.parameters(), lr=1e-2, momentum=0.9)
optim = torch.optim.SGD(model.parameters(), lr = 1e-2, momentum= 0.9)

**What is an Optimizer?**
In machine learning, especially with neural networks, the goal is to find the best set of internal parameters (weights and biases) for your model so that it makes accurate predictions. After the model makes a prediction and you calculate the loss (which quantifies how 'wrong' the prediction was), you need a way to adjust these parameters to reduce that loss in the future. This process of adjusting parameters is called optimization, and the algorithm that does this is called an optimizer.

Think of it like this: you're trying to find the bottom of a valley in a foggy landscape. The loss is your current altitude, and the model's parameters are your current location. An optimizer tells you which direction to step and how big of a step to take to get closer to the bottom.

Explanation of torch.optim.SGD and its parameters:
**torch.optim.SGD:** This specifies that we are using the Stochastic Gradient Descent (SGD) optimizer. SGD is one of the most fundamental and widely used optimization algorithms. It works by iteratively adjusting the model's parameters in the direction opposite to the gradient of the loss function. The 'stochastic' part means it often calculates these gradients on small random subsets of the data (batches) rather than the entire dataset.

**model.parameters():** This argument tells the optimizer which parameters it needs to optimize. When you create a torch.nn.Module (like resnet18), it keeps track of all its learnable parameters (weights and biases). Calling model.parameters() returns an iterator over all these parameters. The optimizer will then update these specific tensors based on the calculated gradients during the backward pass.

**lr=1e-2:** This stands for learning rate. It's a crucial hyperparameter that determines the step size at each iteration while moving towards the minimum of the loss function.
1e-2 means 0.01.
A high learning rate can cause the optimizer to overshoot the minimum, potentially never converging.
A very low learning rate will make the optimization process very slow, taking many steps to reach the minimum.


**momentum=0.9:** Momentum is another hyperparameter used to accelerate SGD in the relevant direction and dampens oscillations. It does this by adding a fraction of the update vector of the past step to the current update vector.
0.9 is a common value for momentum.
It helps the optimizer to overcome local minima and speed up convergence, especially in cases where the loss surface is very flat in one direction and steep in another.

Finally, we call `.step()` to initiate gradient descent. The optimizer
adjusts each parameter by its gradient stored in `.grad`.


In [5]:
optim.step() #gradient descent

At this point, you have everything you need to train your neural
network. The below sections detail the workings of autograd - feel free
to skip them.


------------------------------------------------------------------------


In a NN, parameters that don\'t compute gradients are usually called
**frozen parameters**. It is useful to \"freeze\" part of your model if
you know in advance that you won\'t need the gradients of those
parameters (this offers some performance benefits by reducing autograd
computations).

In finetuning, we freeze most of the model and typically only modify the
classifier layers to make predictions on new labels. Let\'s walk through
a small example to demonstrate this. As before, we load a pretrained
resnet18 model, and freeze all the parameters.


In [6]:
from torch import nn, optim

model = resnet18(weights=ResNet18_Weights.DEFAULT)

# Freeze all the parameters in the network
for param in model.parameters():
    param.requires_grad = False
'''param.requires_grad = False: This is the core of the operation. By setting requires_grad to False for a parameter
 you are telling PyTorch's autograd engine not to compute gradients for that specific parameter during the
 backward pass'''

"param.requires_grad = False: This is the core of the operation. By setting requires_grad to False for a parameter\n you are telling PyTorch's autograd engine not to compute gradients for that specific parameter during the \n backward pass"

Let\'s say we want to finetune the model on a new dataset with 10
labels. In resnet, the classifier is the last linear layer `model.fc`.
We can simply replace it with a new linear layer (unfrozen by default)
that acts as our classifier.


In [7]:
model.fc = nn.Linear(512, 10)

Now all parameters in the model, except the parameters of `model.fc`,
are frozen. The only parameters that compute gradients are the weights
and bias of `model.fc`.


You're exactly right in your understanding! Let's break down model.fc = nn.Linear(512, 10):

model.fc: In many pre-trained models like ResNet, the final layer, which is responsible for classification, is often named fc (fully connected) or classifier. When you assign a new nn.Linear layer to model.fc, you are replacing the original classification head of the pre-trained ResNet18 with a new one.

nn.Linear(512, 10): This creates a new single linear layer, also known as a fully connected layer or a dense layer.

512: This is the input feature dimension. It means this linear layer expects to receive an input tensor with 512 features (or neurons) from the preceding layer of the ResNet model. For ResNet18, the layer before the final fully connected layer typically outputs a 512-dimensional feature vector per image after global average pooling.
10: This is the output feature dimension. It means this linear layer will produce an output tensor with 10 values. In the context of classification, these 10 values correspond to the scores (or logits) for 10 different classes. So, this model is now configured to classify images into 10 distinct categories.

In [8]:
# Optimize only the classifier
optimizer = optim.SGD(model.parameters(), lr=1e-2, momentum=0.9)

------------------------------------------------------------------------
